# TD1 â€” Working with LLMs in Practice
## Module 2IAiasd 9.7 â€” Agentic AI â€” IMT Mines AlÃ¨s

**Objective:** Learn to interact with a local LLM (no API key needed!) using LangChain.  
By the end of this notebook you will have built a **working ReAct agent** that calls Python tools.

**Model used:** `qwen2.5:3b` running locally via [Ollama](https://ollama.com)  

---

### Structure
| # | Exercise | Concepts |
|---|---------|----------|
| 0 | Setup | Ollama, LangChain, LangSmith |
| 1 | Chat basics | `invoke`, `stream`, `batch` |
| 2 | Messages & prompts | `SystemMessage`, `HumanMessage`, few-shot |
| 3 | Structured output | Pydantic, `with_structured_output` |
| 4 | Tool calling | `@tool`, `bind_tools`, manual dispatch |
| 5 | Mini ReAct agent | Full observe â†’ reason â†’ act loop |


---
## Exercise 0 â€” Setup

### 0.1 Install dependencies

In [47]:
# Run once â€” comment out after first execution
%pip install langchain-ollama langchain-core langsmith pydantic --quiet

Note: you may need to restart the kernel to use updated packages.


### 0.2 Check Ollama is running

Open a terminal and run:
```bash
ollama serve          # starts the local server (keep this terminal open)
ollama pull qwen2.5:3b  # download the model (~2 GB, once)
```

Then verify below:

In [48]:
import requests

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    print("âœ… Ollama is running!")
    print(f"Available models: {models}")
except Exception as e:
    print(f"âŒ Ollama not reachable: {e}")
    print("Make sure you ran: ollama serve")

âœ… Ollama is running!
Available models: ['qwen2.5:3b']


### 0.3 (Optional) LangSmith tracing

LangSmith lets you visualise every LLM call, token count, and latency.  
Free account at [smith.langchain.com](https://smith.langchain.com).

If you don't have an account, skip this cell â€” the rest of the notebook works without it.

In [49]:
import os
from dotenv import load_dotenv

# Carrega as variáveis do arquivo .env para o os.environ automaticamente
#load_dotenv(override=True)

#tracing_enabled = os.getenv("LANGCHAIN_TRACING_V2") == "true"
#print(f"LangSmith tracing: {'✅ enabled' if tracing_enabled else '⚠️  disabled (no problem)'}")

### 0.4 Instantiate the model

In [50]:
from langchain_ollama import ChatOllama

# Use qwen2.5:3b (recommended) or qwen2.5:1.5b for low-RAM machines
MODEL = "qwen2.5:3b"

llm = ChatOllama(
    model=MODEL,
    temperature=0,   # deterministic â€” easier to debug
)

# Quick sanity check
response = llm.invoke("Reply with exactly: 'LLM ready!'")
print(response.content)

LLM ready!


---
## Exercise 1 â€” Chat Basics: invoke / stream / batch

LangChain's **Runnable interface** gives every component three calling methods:
- `.invoke()` â€” single call, returns complete response
- `.stream()` â€” returns an iterator of chunks (token by token)
- `.batch()` â€” processes a list of inputs in parallel

### 1.1 invoke()

In [51]:
# .invoke() returns an AIMessage object
response = llm.invoke("What is a Large Language Model? Answer in 2 sentences.")

print("Type:", type(response))
print("Content:", response.content)

# The AIMessage also carries metadata
print("\nMetadata:")
print("  Model:", response.response_metadata.get("model", "n/a"))
print("  Total tokens:", response.usage_metadata)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Type: <class 'langchain_core.messages.ai.AIMessage'>
Content: A Large Language Model is a type of artificial intelligence model designed to understand and generate human-like language, capable of processing and generating text at a scale that can be very large, often trained on vast amounts of text data to improve its language understanding and generation capabilities.

Metadata:
  Model: qwen2.5:3b
  Total tokens: {'input_tokens': 42, 'output_tokens': 53, 'total_tokens': 95}


### 1.2 stream()

In [52]:
# .stream() yields AIMessageChunks one by one
print("Streaming response: ", end="")

full_text = ""
for chunk in llm.stream("List 3 use cases for AI agents in one line each."):
    print(chunk.content, end="", flush=True)
    full_text += chunk.content

print("\n\nTotal characters received:", len(full_text))

Streaming response: 1. Personalized recommendations for e-commerce products.
2. Automated customer service for 24/7 support.
3. Predictive maintenance for industrial machinery.

Total characters received: 156


### 1.3 batch()

In [53]:
import time

questions = [
    "What is RAG? Answer in 1 sentence.",
    "What is ReAct? Answer in 1 sentence.",
    "What is a vector database? Answer in 1 sentence.",
]

# Sequential (slow)
t0 = time.time()
sequential = [llm.invoke(q) for q in questions]
t_seq = time.time() - t0

# Batch (parallel)
t0 = time.time()
batch_results = llm.batch(questions)
t_batch = time.time() - t0

print(f"Sequential: {t_seq:.1f}s | Batch: {t_batch:.1f}s")
print(f"Speedup: {t_seq/t_batch:.1f}x\n")

for q, r in zip(questions, batch_results):
    print(f"Q: {q}")
    print(f"A: {r.content}\n")

Sequential: 0.8s | Batch: 0.8s
Speedup: 1.0x

Q: What is RAG? Answer in 1 sentence.
A: RAG stands for Retrieval-Augmented Generation, a method that uses pre-existing documents to augment the generation process, often used in natural language processing and AI systems.

Q: What is ReAct? Answer in 1 sentence.
A: ReAct is a framework for integrating external memory into transformer models, allowing them to use and manipulate external data more effectively.

Q: What is a vector database? Answer in 1 sentence.
A: A vector database is a type of database that stores and retrieves data based on semantic similarity using vector representations of data points.



### âœï¸ Exercise 1 â€” Your turn

Use `.batch()` to ask the model to translate 4 short phrases to French.  
Time it and compare to sequential.

In [ ]:
# TODO: your code here
phrases = [
    "Translate to French: 'The model is thinking.'",
    "Translate to French: 'The agent uses tools.'",
    "Translate to French: 'Memory is essential.'",
    "Translate to French: 'Retrieval augments generation.'",
]

# Sequential (slow)
t0 = time.time()
sequential = [llm.invoke(ph) for ph in phrases]
t_seq = time.time() - t0

# Batch (parallel)
t0 = time.time()
batch_results = llm.batch(phrases)
t_batch = time.time() - t0

print(f"Sequential: {t_seq:.1f}s | Batch: {t_batch:.1f}s")
print(f"Speedup: {t_seq/t_batch:.1f}x\n")

for q, r in zip(phrases, batch_results):
    print(f"Q: {q}")
    print(f"A: {r.content}\n")


Sequential: 0.4s | Batch: 0.4s
Speedup: 1.0x

Q: Translate to French: 'The model is thinking.'
A: La modèle pense.

Q: Translate to French: 'The agent uses tools.'
A: L'agent utilise des outils.

Q: Translate to French: 'Memory is essential.'
A: La mémoire est essentielle.

Q: Translate to French: 'Retrieval augments generation.'
A: "Retrieval augmente la génération."



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


---
## Exercise 2 â€” Messages & System Prompts

The **role** of a message determines how the model interprets it:
- `SystemMessage` â€” defines the model's persona, rules, output format
- `HumanMessage` â€” the user's input
- `AIMessage` â€” the model's response (can be used for few-shot examples)

In [55]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Basic system prompt
messages = [
    SystemMessage(content="You are a concise technical assistant. "
                           "Always answer in bullet points, max 3 bullets."),
    HumanMessage(content="What are the main components of an AI agent?"),
]

response = llm.invoke(messages)
print(response.content)

- Perception System
- Cognitive Engine
- Action Selector


### 2.1 Few-shot prompting with AIMessage

Include example exchanges to teach the model the **exact format** you want.

In [56]:
# Few-shot: teach the model to classify sentiment with a fixed format
few_shot_messages = [
    SystemMessage(content="Classify the sentiment of the input. "
                           "Reply with exactly: LABEL: <positive|negative|neutral>"),
    # Example 1
    HumanMessage(content="I love this product!"),
    AIMessage(content="LABEL: positive"),
    # Example 2
    HumanMessage(content="The delivery was terrible."),
    AIMessage(content="LABEL: negative"),
    # Example 3
    HumanMessage(content="It arrived yesterday."),
    AIMessage(content="LABEL: neutral"),
    # New examples
    HumanMessage(content="I really enjoy using this app."),
    AIMessage(content="LABEL: positive"),
    HumanMessage(content="I dislike the new update."),
    AIMessage(content="LABEL: negative"),
    HumanMessage(content="Rain doesn't bother me."),
    AIMessage(content="LABEL: neutral"),
    # New input
    HumanMessage(content="The interface is confusing but the features are great."),
]

response = llm.invoke(few_shot_messages)
print("Response:", response.content)

# Compare: without few-shot
no_few_shot = [
    SystemMessage(content="Classify the sentiment."),
    HumanMessage(content="The interface is confusing but the features are great."),
]
response2 = llm.invoke(no_few_shot)
print("Without few-shot:", response2.content)

Response: LABEL: neutral
Without few-shot: The sentiment in this statement is mixed. There is a negative aspect (confusing interface) and a positive aspect (great features).


### 2.2 PromptTemplate â€” reusable prompts

In [57]:
from langchain_core.prompts import ChatPromptTemplate

# Define a reusable template
template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. "
               "Explain concepts clearly and give one practical example."),
    ("human", "Explain: {concept}"),
])

# Chain the template with the LLM using the | operator (LCEL)
chain = template | llm

# Invoke with variables
result = chain.invoke({
    "domain": "machine learning",
    "concept": "overfitting",
})
print(result.content)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Overfitting is a common issue in machine learning where a model learns the detail and noise in the training data to an extent that it negatively impacts the performance of the model on new, unseen data. Essentially, an overfit model captures the noise in the training data, which means it performs very well on the training data but poorly on the new, unseen data.

### Key Characteristics of Overfitting:
1. **High Training Accuracy**: The model performs exceptionally well on the training data.
2. **Poor Generalization**: The model fails to generalize well to new data.
3. **Complexity**: Often occurs with complex models that have many parameters, which can lead to capturing noise as patterns.

### Example:
Imagine you are training a machine learning model to predict house prices based on features like the number of rooms, location, and age of the house. If you have a very complex model (e.g., a deep neural network with many layers and neurons), it might start to learn not just the underly

### âœï¸ Exercise 2 â€” Your turn

Create a `ChatPromptTemplate` for a **code review assistant** that:
- Takes `language` and `code` as variables
- Returns a structured review with: bugs found, improvements, quality score (1-5)

In [ ]:
# TODO: your code here
code_to_review = """
def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n)  # BUG: should be factorial(n-1)
"""

# Define a reusable template
template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in this programming language: {language}. "
               "You will review the code and provide feedback on potential bugs, "
               "improvements, and best practices. It is also importante to start with a "
               "brief summary of the code's purpose and functionality."
               "At the end, rate it from 1 to 5 stars, where 5 is the best."),
    ("human", "Explain: {code}"),
])

# Chain the template with the LLM using the | operator (LCEL)
chain = template | llm

# Invoke with variables
result = chain.invoke({
    "language": "Python",
    "code": code_to_review,
})
print(result.content)


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


The provided code is intended to calculate the factorial of a number `n`, which is the product of all positive integers less than or equal to `n`. However, there is a bug in the recursive implementation. The recursive call should be `factorial(n-1)` instead of `factorial(n)`. This is because the factorial of `n` is `n * factorial(n-1)`, not `n * factorial(n)`.

Here is the corrected version of the function:

```python
def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n - 1)
```

### Explanation of the Corrected Function:
- The function checks if `n` is 0. If it is, it returns 1, as the factorial of 0 is defined as 1.
- If `n` is not 0, the function calls itself with `n-1` and multiplies the result by `n`. This recursive call continues until `n` reaches 0, at which point the recursion unwinds, multiplying all the way back up to the original value of `n`.

### Functionality Summary:
- The function correctly calculates the factorial of a number using recursion.
- 

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


---
## Exercise 3 â€” Structured Outputs

Raw text is unreliable for programs. Use Pydantic schemas to force the model to return structured, typed data.

In [59]:
from pydantic import BaseModel, Field
from typing import Literal

# Define the output schema
class SentimentAnalysis(BaseModel):
    """Sentiment analysis result for a piece of text."""
    label: Literal["positive", "negative", "neutral"] = Field(
        description="Overall sentiment label"
    )
    confidence: float = Field(
        description="Confidence score between 0.0 and 100.0",
        ge=0.0,
        le=100.0,
    )
    key_phrases: list[str] = Field(
        description="List of 1-3 key phrases that drove the classification"
    )

# Bind the schema to the LLM
structured_llm = llm.with_structured_output(SentimentAnalysis)

# Test
result = structured_llm.invoke(
    "The new update broke half the features but the UI looks stunning."
)

print(f"Label:       {result.label}")
print(f"Confidence:  {result.confidence:.2f}%")
print(f"Key phrases: {result.key_phrases}")
print(f"Type:        {type(result)}")

Label:       negative
Confidence:  95.00%
Key phrases: ['new update', 'broke half the features', 'stunning UI']
Type:        <class '__main__.SentimentAnalysis'>


### 3.1 More complex schemas

In [60]:
from pydantic import BaseModel, Field
from typing import Optional

class CodeReview(BaseModel):
    """A structured code review."""
    has_bugs: bool = Field(description="True if bugs were found")
    bugs: list[str] = Field(description="List of bug descriptions, empty if none")
    improvements: list[str] = Field(description="Suggested improvements")
    quality_score: int = Field(description="Code quality from 1 (poor) to 5 (excellent)",
                                ge=1, le=5)
    summary: str = Field(description="One-sentence overall assessment")

reviewer = llm.with_structured_output(CodeReview)

code = """
def get_average(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)  # will crash on empty list
"""

review = reviewer.invoke(
    f"Review this Python code:\n{code}"
)

print(f"Has bugs:    {review.has_bugs}")
print(f"Bugs:        {review.bugs}")
print(f"Improvements:{review.improvements}")
print(f"Score:       {review.quality_score}/5")
print(f"Summary:     {review.summary}")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Has bugs:    True
Bugs:        ['The code will crash if the list is empty because you are trying to divide by the length of the list, which will result in a ZeroDivisionError. You should add a check to handle the case where the list is empty.']
Improvements:['Add a check to see if the list is empty and return 0 in that case.']
Score:       3/5
Summary:     The code has a bug related to division by zero. It should handle the case where the list is empty.


### âœï¸ Exercise 3 â€” Your turn

Create a Pydantic schema for **entity extraction**: given a sentence, extract all persons, organisations and locations mentioned.

In [ ]:
test_sentence = """Emmanuel Macron met Elon Musk at the Elysée Palace in Paris
to discuss Tesla's expansion in France and SpaceX's satellite programme. Patrick Jane
and Lisbon also met at Lisbon to drink a tea at the Lisbon Palace."""

class EntityExtraction(BaseModel):
    """Extract entities and confidence levels from text."""
    people: list[tuple[str, float]] = Field(
        description="List of tuples containing (person_name, confidence_score_0_to_100), empty if none"
    )
    organisations: list[tuple[str, float]] = Field(
        description="List of tuples containing (organisation_name, confidence_score_0_to_100), empty if none"
    )
    locations: list[tuple[str, float]] = Field(
        description="List of tuples containing (location_name, confidence_score_0_to_100), empty if none"
    )

extractor = llm.with_structured_output(EntityExtraction)

extraction = extractor.invoke(
    "Extract all entities in the following sentence with confidence scores from 0 to 100:"
    f"\n{test_sentence}"
)

# Opção 1: Formatação direta mantendo o formato de lista
print("People:       ", [f"{entity} ({conf:.0f}%)" for entity, conf in extraction.people])
print("Organisations:", [f"{entity} ({conf:.0f}%)" for entity, conf in extraction.organisations])
print("Locations:    ", [f"{entity} ({conf:.0f}%)" for entity, conf in extraction.locations])

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


People:        ['Emmanuel Macron (100%)', 'Elon Musk (100%)', 'Patrick Jane (100%)', 'Lisbon (100%)']
Organisations: []
Locations:     ['Elysée Palace (100%)', 'Paris (100%)', 'Lisbon Palace (100%)', 'Lisbon (100%)']


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


---
## Exercise 4 â€” Tool Calling

**Tool calling** lets the LLM decide when to call a Python function instead of generating text.  
The model never executes the function â€” it just tells you **which function to call and with what arguments**.

### 4.1 Defining tools with @tool

In [62]:
from langchain_core.tools import tool

# The docstring is what the LLM reads to understand the tool.
# Write it clearly â€” it's your prompt to the model about this tool.

@tool
def get_word_count(text: str) -> int:
    """Count the number of words in a given text string.

    Args:
        text (str): The input text to analyze.

    Returns:
        int: The number of words in the input text.
    """
    return len(text.split())

@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature value from Celsius to Fahrenheit."""
    return round(celsius * 9/5 + 32, 2)

@tool
def get_current_date() -> str:
    """Return today's date in ISO format (YYYY-MM-DD)."""
    from datetime import date
    return date.today().isoformat()

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """Calculate Body Mass Index (BMI) given weight in kg and height in metres."""
    return round(weight_kg / (height_m ** 2), 2)

# Inspect what the LLM sees
print("Tool name:", get_word_count.name)
print("Tool description:", get_word_count.description)
print("Tool schema:", get_word_count.args_schema.model_json_schema())

Tool name: get_word_count
Tool description: Count the number of words in a given text string.

    Args:
        text (str): The input text to analyze.

    Returns:
        int: The number of words in the input text.
Tool schema: {'description': 'Count the number of words in a given text string.\n\nArgs:\n    text (str): The input text to analyze.\n\nReturns:\n    int: The number of words in the input text.', 'properties': {'text': {'title': 'Text', 'type': 'string'}}, 'required': ['text'], 'title': 'get_word_count', 'type': 'object'}


### 4.2 Binding tools and making a call

In [63]:
tools = [get_word_count, celsius_to_fahrenheit, get_current_date, calculate_bmi]

# Bind tools to the LLM
llm_with_tools = llm.bind_tools(tools)

# Ask something that requires a tool
response = llm_with_tools.invoke(
    "How many words are in the sentence: 'Artificial intelligence is transforming every industry'?"
)

print("Response type:", type(response))
print("Content:", response.content)            # empty string â€” model wants to call a tool
print("Tool calls:", response.tool_calls)       # list of {name, args, id}

Response type: <class 'langchain_core.messages.ai.AIMessage'>
Content: 
Tool calls: [{'name': 'get_word_count', 'args': {'text': 'Artificial intelligence is transforming every industry'}, 'id': '130c6835-d381-4224-b96c-31a885f0e0e0', 'type': 'tool_call'}]


### 4.3 The tool dispatch loop

In [64]:
from langchain_core.messages import ToolMessage

def run_tool(tool_call: dict, tools_map: dict) -> ToolMessage:
    """Execute a tool call and return a ToolMessage."""
    fn = tools_map[tool_call["name"]]
    result = fn.invoke(tool_call["args"])
    return ToolMessage(
        content=str(result),
        tool_call_id=tool_call["id"],
    )

# Build a lookup map
tools_map = {t.name: t for t in tools}

# --- Single-turn tool call ---
query = "What is 38 degrees Celsius in Fahrenheit?"
messages = [HumanMessage(content=query)]

# Step 1: Model decides to call a tool
response = llm_with_tools.invoke(messages)
messages.append(response)
print(f"Tool call: {response.tool_calls}")

# Step 2: Execute the tool(s)
for tc in response.tool_calls:
    tool_msg = run_tool(tc, tools_map)
    messages.append(tool_msg)
    print(f"Tool result: {tool_msg.content}")

# Step 3: Model generates the final answer
final = llm_with_tools.invoke(messages)
print(f"\nFinal answer: {final.content}")

Tool call: [{'name': 'celsius_to_fahrenheit', 'args': {'celsius': 38}, 'id': '61109f9f-986e-49e5-b60a-86c9702beae9', 'type': 'tool_call'}]
Tool result: 100.4

Final answer: 38 degrees Celsius is 100.4 degrees Fahrenheit.


### 4.4 Testing with different queries

In [65]:
def single_tool_query(query: str) -> str:
    """Run a single query through the tool-calling loop."""
    messages = [HumanMessage(content=query)]
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        return response.content  # no tool needed

    for tc in response.tool_calls:
        messages.append(run_tool(tc, tools_map))

    final = llm_with_tools.invoke(messages)
    return final.content

test_queries = [
    "What is today's date?",
    "What is the BMI of someone who weighs 70 kg and is 1.75 m tall?",
    "How many words in: 'The quick brown fox jumps over the lazy dog'?",
    "What is the capital of France?",  # no tool needed
]

for q in test_queries:
    answer = single_tool_query(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

Q: What is today's date?
A: Today's date is October 16, 2026.

Q: What is the BMI of someone who weighs 70 kg and is 1.75 m tall?
A: The Body Mass Index (BMI) of someone who weighs 70 kg and is 1.75 m tall is approximately 22.86.

Q: How many words in: 'The quick brown fox jumps over the lazy dog'?
A: The sentence 'The quick brown fox jumps over the lazy dog' contains 9 words.

Q: What is the capital of France?
A: The capital of France is Paris. Is there a specific information you need about Paris, such as its population, area, or other details?



### âœï¸ Exercise 4 â€” Your turn

Create 2 new tools of your choice (ideas: `string_reverse`, `is_palindrome`, `count_vowels`, `days_until_christmas`, `euro_to_cfa`) and test them.

In [ ]:
# TODO: define 2 tools and test them

@tool
def string_reverse(text: str) -> str:
    """Reverse the input string.

    Args:
        text (str): The string to reverse.

    Returns:
        str: The reversed string.
    """
    return text[::-1]


@tool
def celsius_to_kelvin(celsius: float) -> float:
    """Convert a temperature value from Celsius to Kelvin.

    Args:
        celsius (float): The temperature in degrees Celsius.

    Returns:
        float: The temperature in Kelvin.
    """
    return celsius + 273.15

# Add your tools to the list and bind to the LLM
tools.append(string_reverse)
tools.append(celsius_to_kelvin)

llm_with_tools = llm.bind_tools(tools)

# Updating the tools_map with the new tools
tools_map = {t.name: t for t in tools}

# Test the new tools with some queries
test_queries = [
    "How the string: 'Hello, world!' looks when reversed?",
    "What is 25 degrees Celsius in Kelvin?",
    "What is Corinthians?",  # no tool needed
]

for q in test_queries:
    answer = single_tool_query(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

Q: How the string: 'Hello, world!' looks when reversed?
A: The string 'Hello, world!' reversed is '!dlrow ,olleH'.

Q: What is 25 degrees Celsius in Kelvin?
A: 25 degrees Celsius is equivalent to 298.15 Kelvin.

Q: What is Corinthians?
A: Could you please provide more context or specify what you want to know about Corinthians? Is it a sports team, a person's name, or something else?



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


---
## Exercise 5 â€” Mini ReAct Agent

A **ReAct agent** (Reason + Act) loops until the model stops requesting tool calls.  
This is the foundation of **every** agent framework (LangGraph, CrewAI, AutoGen...).

```
User query
    â”‚
    â–¼
LLM reasons â”€â”€â–º has tool calls? â”€â”€YESâ”€â”€â–º execute tools â”€â”€â–º append result â”€â”€â”
    â”‚                                                                        â”‚
    NO                                                                       â”‚
    â”‚â—„â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
    â–¼
Final answer
```

### 5.1 Math tools

In [67]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_ollama import ChatOllama

# -- Tool definitions --
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b. Raises an error if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

@tool
def power(base: float, exponent: float) -> float:
    """Raise base to the power of exponent."""
    return base ** exponent

agent_tools = [add, subtract, multiply, divide, power]
agent_tools_map = {t.name: t for t in agent_tools}

print(f"Tools available: {[t.name for t in agent_tools]}")

Tools available: ['add', 'subtract', 'multiply', 'divide', 'power']


### 5.2 The ReAct loop

In [68]:
def react_agent(query: str, verbose: bool = True) -> str:
    """
    A minimal ReAct agent.
    Loops until the LLM produces a response with no tool calls.
    """
    agent_llm = ChatOllama(model=MODEL, temperature=0).bind_tools(agent_tools)

    messages = [
        SystemMessage(content="You are a helpful math assistant. "
                               "Use the provided tools to solve calculations. "
                               "Show your reasoning step by step."),
        HumanMessage(content=query),
    ]

    step = 0
    max_steps = 10  # safety limit

    while step < max_steps:
        step += 1
        response = agent_llm.invoke(messages)
        messages.append(response)

        if verbose:
            print(f"\n--- Step {step} ---")
            if response.content:
                print(f"  Thought: {response.content}")
            if response.tool_calls:
                for tc in response.tool_calls:
                    print(f"  Action:  {tc['name']}({tc['args']})")

        # No tool calls â†’ final answer
        if not response.tool_calls:
            if verbose:
                print(f"\nâœ… Final answer: {response.content}")
            return response.content

        # Execute all tool calls and append results
        for tc in response.tool_calls:
            try:
                result = agent_tools_map[tc["name"]].invoke(tc["args"])
                tool_msg = ToolMessage(
                    content=str(result),
                    tool_call_id=tc["id"],
                )
                if verbose:
                    print(f"  Observation: {tc['name']} = {result}")
            except Exception as e:
                tool_msg = ToolMessage(
                    content=f"Error: {e}",
                    tool_call_id=tc["id"],
                )
                if verbose:
                    print(f"  âš ï¸ Error: {e}")
            messages.append(tool_msg)

    return "Max steps reached â€” agent did not converge."


# Test 1: simple multi-step calculation
result = react_agent("What is (3 + 5) * 12?")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



--- Step 1 ---
  Thought: To solve the expression (3 + 5) * 12, we first need to perform the operation inside the parentheses and then multiply the result by 12.

  Action:  add({'a': 3, 'b': 5})
  Action:  add({'a': 8, 'b': 12})
  Observation: add = 8.0
  Observation: add = 20.0


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



--- Step 2 ---
  Action:  add({'a': 3, 'b': 5})
  Action:  multiply({'a': 12, 'b': 4})
  Observation: add = 8.0
  Observation: multiply = 48.0


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



--- Step 3 ---
  Thought: It seems there was a misunderstanding in the previous steps. Let's correct it and proceed with the calculation:

1. First, we need to add 3 and 5, which gives us 8.
2. Then, we multiply the result by 12.

Let's perform these steps using the provided functions.

  Action:  add({'a': 3, 'b': 5})
  Action:  multiply({'a': 12, 'b': 4})
  Observation: add = 8.0
  Observation: multiply = 48.0

--- Step 4 ---
  Action:  add({'a': 3, 'b': 5})
  Action:  multiply({'a': 12, 'b': 4})
  Observation: add = 8.0
  Observation: multiply = 48.0


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



--- Step 5 ---
  Thought: You're correct, let's ensure we're following the steps accurately:

1. Add 3 and 5: 
   - Result: 8
2. Multiply the result by 12:
   - Result: 8 * 12 = 96

It appears there was a mistake in the previous steps. Let's perform the calculation again:


  Action:  add({'a': 3, 'b': 5})
  Action:  multiply({'a': 12, 'b': 8})
  Observation: add = 8.0
  Observation: multiply = 96.0

--- Step 6 ---
  Action:  add({'a': 3, 'b': 5})
  Action:  multiply({'a': 12, 'b': 8})
  Observation: add = 8.0
  Observation: multiply = 96.0

--- Step 7 ---
  Action:  add({'a': 3, 'b': 5})
  Action:  multiply({'a': 12, 'b': 8})
  Observation: add = 8.0
  Observation: multiply = 96.0

--- Step 8 ---
  Action:  add({'a': 3, 'b': 5})
  Action:  multiply({'a': 12, 'b': 8})
  Observation: add = 8.0
  Observation: multiply = 96.0


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



--- Step 9 ---
  Thought: The calculation is now correct. Here's the step-by-step breakdown:

1. Add 3 and 5:
   - Result: 8
2. Multiply the result by 12:
   - Result: 8 * 12 = 96

Therefore, the final answer to the expression (3 + 5) * 12 is 96.

âœ… Final answer: The calculation is now correct. Here's the step-by-step breakdown:

1. Add 3 and 5:
   - Result: 8
2. Multiply the result by 12:
   - Result: 8 * 12 = 96

Therefore, the final answer to the expression (3 + 5) * 12 is 96.


In [69]:
# Test 2: more complex
result = react_agent("Calculate: 2 to the power of 10, then divide by 4, then subtract 200.")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



--- Step 1 ---
  Action:  power({'base': 2, 'exponent': 10})
  Action:  power({'a': 1024, 'b': 4})
  Action:  subtract({'a': 256, 'b': 200})
  Action:  subtract({'a': 56, 'b': 200})
  Action:  subtract({'a': -144, 'b': 200})
  Observation: power = 1024.0
  âš ï¸ Error: 2 validation errors for power
base
  Field required [type=missing, input_value={'a': 1024, 'b': 4}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
exponent
  Field required [type=missing, input_value={'a': 1024, 'b': 4}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
  Observation: subtract = 56.0
  Observation: subtract = -144.0
  Observation: subtract = -344.0


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



--- Step 2 ---
  Thought: It seems there was a misunderstanding in the sequence of operations. Let's correct the steps and calculate it properly:

1. Calculate 2 to the power of 10.
2. Divide the result by 4.
3. Subtract 200 from the result.

Let's proceed with these steps.

  Action:  power({'base': 2, 'exponent': 10})
  Action:  divide({'a': 1024, 'b': 4})
  Action:  subtract({'b': 200, 'a': 256})
  Observation: power = 1024.0
  Observation: divide = 256.0
  Observation: subtract = 56.0


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



--- Step 3 ---
  Action:  power({'base': 2, 'exponent': 10})
  Action:  divide({'a': 1024, 'b': 4})
  Action:  subtract({'a': 256, 'b': 200})
  Action:  subtract({'b': 200, 'a': 56})
  Action:  subtract({'a': -144, 'b': 200})
  Observation: power = 1024.0
  Observation: divide = 256.0
  Observation: subtract = 56.0
  Observation: subtract = -144.0
  Observation: subtract = -344.0


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



--- Step 4 ---
  Thought: It seems there is a persistent misunderstanding in the sequence of operations. Let's correct the steps and calculate it properly:

1. Calculate \(2^{10}\).
2. Divide the result by 4.
3. Subtract 200 from the result.

Let's perform these steps:

1. Calculate \(2^{10}\):

  Action:  power({'base': 2, 'exponent': 10})
  Observation: power = 1024.0


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



--- Step 5 ---
  Thought: The calculation for \(2^{10}\) is 1024.0.

2. Divide 1024 by 4:

  Action:  divide({'a': 1024, 'b': 4})
  Observation: divide = 256.0

--- Step 6 ---
  Thought: The calculation for dividing 1024 by 4 is 256.0.

3. Subtract 200 from 256:


âœ… Final answer: The calculation for dividing 1024 by 4 is 256.0.

3. Subtract 200 from 256:



In [70]:
# Test 3: question that doesn't need tools
result = react_agent("What is the Pythagorean theorem?", verbose=True)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')



--- Step 1 ---
  Thought: The Pythagorean theorem is a fundamental principle in geometry that relates to right-angled triangles. It states that the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides. This can be mathematically represented as \(c^2 = a^2 + b^2\), where \(c\) is the length of the hypotenuse, and \(a\) and \(b\) are the lengths of the other two sides.

While I can't directly calculate the Pythagorean theorem here, I can demonstrate how to calculate a specific example using the provided functions. Let's calculate \(c\) for a right triangle with sides \(a = 3\) and \(b = 4\).

âœ… Final answer: The Pythagorean theorem is a fundamental principle in geometry that relates to right-angled triangles. It states that the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides. This can be mathe

### 5.3 Mixed tools: maths + string + date

In [ ]:
# Combine maths tools with the utility tools from Exercise 4
mixed_tools = agent_tools + [get_current_date, get_word_count, celsius_to_fahrenheit]
mixed_tools_map = {t.name: t for t in mixed_tools}

def mixed_agent(query: str) -> str:
    """ReAct agent with mixed tools."""
    mixed_llm = ChatOllama(model=MODEL, temperature=0).bind_tools(mixed_tools)
    messages = [HumanMessage(content=query)]

    for _ in range(8):
        response = mixed_llm.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            return response.content
        for tc in response.tool_calls:
            result = mixed_tools_map[tc["name"]].invoke(tc["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    return "Max iterations reached."


queries = [
    "Today's date, and how many words in 'Hello world from IMT Mines Ales'?",
    "What is 100 Celsius in Fahrenheit, and what is that number divided by 3?",
]

for q in queries:
    print(f"Q: {q}")
    print(f"A: {mixed_agent(q)}\n")

Q: Today's date, and how many words in 'Hello world from IMT Mines Ales'?


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


A: Today's date is 2026-10-16 and the text 'Hello world from IMT Mines Ales' contains 6 words.

Q: What is 100 Celsius in Fahrenheit, and what is that number divided by 3?
A: The temperature of 100 degrees Celsius is 212 degrees Fahrenheit. When you divide that number by 3, you get approximately 33.33.



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


---
## Summary & Key Takeaways

| Concept | What you learned | LangChain API |
|---------|-----------------|---------------|
| Chat interface | 3 calling modes | `.invoke()` / `.stream()` / `.batch()` |
| Messages | Role-based conversation | `SystemMessage`, `HumanMessage`, `AIMessage` |
| Prompts | Templates & few-shot | `ChatPromptTemplate` |
| Structured output | Typed, reliable extraction | `.with_structured_output(Schema)` |
| Tool calling | LLM â†’ function dispatch | `@tool`, `.bind_tools()`, `ToolMessage` |
| ReAct agent | Observe â†’ Reason â†’ Act loop | Manual loop (LangGraph next session) |

---
## Bonus â€” Inspect your LangSmith traces

If you enabled LangSmith tracing (Exercise 0.3), go to [smith.langchain.com](https://smith.langchain.com) and open project **2IAiasd-9.7-TD1**.  
You should see every LLM call, with:
- Input messages
- Output message + tool calls
- Token count and latency
- Full trace tree for the agent loop

This is exactly what you'll use to debug your project agent in EDT3.